# Task B (Colab Edition)

Self-contained SOC automation helper: load the trained Task A model, classify emails, and extract indicators of compromise without importing repository modules.

In [ ]:
%%capture
!pip install -q pandas numpy joblib beautifulsoup4 lxml

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

import joblib
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
MODEL_PATH = ARTIFACT_DIR / "best_model.joblib"
FALLBACK_PATH = ARTIFACT_DIR / "ml" / "logistic_regression.joblib"

print(f"Artifacts directory: {ARTIFACT_DIR}")
if not MODEL_PATH.exists() and not FALLBACK_PATH.exists():
    raise FileNotFoundError("Upload best_model.joblib (from Task A) into artifacts/ before running this notebook.")

In [ ]:
# -----------------------------------------------------------------------------
# Text cleaning + IOC extraction helpers
# -----------------------------------------------------------------------------

import re


def strip_html(text: Optional[str]) -> str:
    if not text:
        return ''
    soup = BeautifulSoup(text, 'lxml')
    return soup.get_text(separator=' ').strip()


def normalize_text(text: Optional[str]) -> str:
    if not text:
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+', ' <URL> ', text)
    text = re.sub(r'[\w\.-]+@[\w\.-]+', ' <EMAIL> ', text)
    text = re.sub(r'\b\d+(?:\.\d+)?\b', ' <NUMBER> ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


IOC_PATTERNS = {
    'urls': re.compile(r'https?://[\w\-./?=&%]+', re.IGNORECASE),
    'ips': re.compile(r'(?:\d{1,3}\.){3}\d{1,3}'),
    'domains': re.compile(r'(?:[a-z0-9-]+\.)+[a-z]{2,}', re.IGNORECASE),
    'emails': re.compile(r'[\w\.-]+@[\w\.-]+\.[a-z]{2,}', re.IGNORECASE),
}


def extract_iocs(text: str) -> Dict[str, List[str]]:
    findings: Dict[str, List[str]] = {}
    for key, pattern in IOC_PATTERNS.items():
        matches = pattern.findall(text)
        if matches:
            seen = []
            for item in matches:
                if item not in seen:
                    seen.append(item)
            findings[key] = seen
    return findings


def summarize_iocs(iocs: Dict[str, List[str]]) -> Dict[str, int]:
    summary = {key: len(values) for key, values in iocs.items()}
    summary['total'] = sum(summary.values())
    return summary

In [ ]:
# -----------------------------------------------------------------------------
# Model loading + inference helpers
# -----------------------------------------------------------------------------

_MODEL_CACHE = None


def _load_model():
    global _MODEL_CACHE
    if _MODEL_CACHE is not None:
        return _MODEL_CACHE
    if MODEL_PATH.exists():
        _MODEL_CACHE = joblib.load(MODEL_PATH)
    elif FALLBACK_PATH.exists():
        _MODEL_CACHE = joblib.load(FALLBACK_PATH)
    else:
        raise FileNotFoundError('No trained model available.')
    return _MODEL_CACHE


def classify_text(text: str, is_html: bool = False) -> Dict[str, Any]:
    model = _load_model()
    cleaned = normalize_text(strip_html(text) if is_html else text)
    if hasattr(model, 'predict_proba'):
        prob = model.predict_proba([cleaned])[:, 1]
    else:
        decision = model.decision_function([cleaned])
        prob = 1 / (1 + np.exp(-decision))
    probability = float(prob[0])
    label = int(probability >= 0.5)
    risk_level = _derive_risk_level(probability)
    confidence = float(abs(probability - 0.5) * 2)
    explanation = _explain(model, cleaned, label, probability)
    recommendations = _recommend(label, risk_level)
    return {
        'label': label,
        'score': probability,
        'risk_level': risk_level,
        'confidence': confidence,
        'explanations': explanation,
        'recommendations': recommendations,
    }


def reload_model() -> None:
    global _MODEL_CACHE
    _MODEL_CACHE = None


def _derive_risk_level(prob: float) -> str:
    if prob >= 0.85:
        return 'high'
    if prob >= 0.65:
        return 'elevated'
    if prob >= 0.45:
        return 'moderate'
    return 'low'


def _explain(model: Any, text: str, label: int, probability: float) -> Dict[str, Any]:
    explanation: Dict[str, Any] = {'method': 'tfidf'}
    if not hasattr(model, 'named_steps'):
        explanation['rationale'] = _compose_rationale(label, probability, [])
        return explanation
    vectorizer = model.named_steps.get('tfidf')
    if vectorizer is None:
        explanation['rationale'] = _compose_rationale(label, probability, [])
        return explanation
    feature_names = vectorizer.get_feature_names_out()
    vector = vectorizer.transform([text]).toarray()[0]
    clf = model.named_steps.get('clf')
    top_indices = np.argsort(vector)[::-1]
    top_terms = [
        {'term': feature_names[idx], 'weight': float(vector[idx])}
        for idx in top_indices[:8]
        if vector[idx] > 0
    ]
    explanation['top_terms'] = top_terms
    if clf is not None and hasattr(clf, 'coef_'):
        coef = np.asarray(clf.coef_)[0]
        contributions = vector * coef
        pos = np.argsort(contributions)[::-1]
        neg = np.argsort(contributions)
        explanation['method'] = 'linear_coefficients'
        explanation['supporting_terms'] = [
            {'term': feature_names[idx], 'contribution': float(contributions[idx])}
            for idx in pos[:5]
            if contributions[idx] > 0
        ]
        explanation['mitigating_terms'] = [
            {'term': feature_names[idx], 'contribution': float(contributions[idx])}
            for idx in neg[:5]
            if contributions[idx] < 0
        ]
        highlights = explanation.get('supporting_terms') or top_terms
    else:
        highlights = top_terms
    explanation['rationale'] = _compose_rationale(label, probability, highlights)
    return explanation


def _compose_rationale(label: int, probability: float, highlights: List[Dict[str, Any]]) -> str:
    status = 'phishing' if label == 1 else 'legitimate'
    confidence = f"{probability * 100:.1f}%"
    if not highlights:
        return f"Model judges the message as {status} with {confidence} confidence."
    tokens = ', '.join(item['term'] for item in highlights[:3])
    return f"Model judges the message as {status} with {confidence} confidence, driven by tokens: {tokens}."


def _recommend(label: int, risk: str) -> List[str]:
    if label == 1:
        actions = [
            'Quarantine the message and block the sender domain.',
            'Open a phishing investigation ticket with the SOC platform.',
        ]
        if risk in {'high', 'elevated'}:
            actions.append('Trigger user password reset if credentials may be exposed.')
        return actions
    guidance = ['Log the event for monitoring.']
    if risk in {'moderate', 'elevated'}:
        guidance.append('Consider manual review before releasing the message.')
    return guidance

In [ ]:
# -----------------------------------------------------------------------------
# SOC inference dataclass + examples
# -----------------------------------------------------------------------------

@dataclass
class SOCInference:
    subject: str
    body: str
    is_html: bool = False

    def run(self) -> Dict[str, Any]:
        combined = f"{self.subject}

{self.body}".strip()
        model_output = classify_text(combined, is_html=self.is_html)
        iocs = extract_iocs(combined)
        return {
            'subject': self.subject,
            'risk_level': model_output['risk_level'],
            'label': model_output['label'],
            'probability': model_output['score'],
            'confidence': model_output['confidence'],
            'rationale': model_output['explanations'].get('rationale'),
            'top_terms': model_output['explanations'].get('supporting_terms') or model_output['explanations'].get('top_terms'),
            'recommendations': model_output['recommendations'],
            'ioc_summary': summarize_iocs(iocs),
            'iocs': iocs,
        }

examples = [
    {
        'subject': 'Payroll update',
        'body': 'Hi team, please review the attached payroll spreadsheet before Friday. Thanks!',
    },
    {
        'subject': 'URGENT: Reset your account password now',
        'body': 'Your account has been compromised. Visit http://secure-login-example.com immediately to verify your credentials or your account will be terminated.',
    },
]

outputs = [SOCInference(**example).run() for example in examples]
outputs

In [ ]:
# -----------------------------------------------------------------------------
# Batch scoring helper for CSV exports
# -----------------------------------------------------------------------------

def score_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for row in df.itertuples(index=False):
        subject = getattr(row, 'subject', '')
        body = getattr(row, 'body', '')
        is_html = bool(getattr(row, 'body_is_html', False))
        enriched = SOCInference(subject=subject, body=body, is_html=is_html).run()
        record = {
            'subject': enriched['subject'],
            'label': enriched['label'],
            'probability': enriched['probability'],
            'risk_level': enriched['risk_level'],
            'confidence': enriched['confidence'],
            'ioc_total': enriched['ioc_summary'].get('total', 0),
            'recommendations': ' | '.join(enriched['recommendations']),
        }
        records.append(record)
    return pd.DataFrame(records)

sample_df = pd.DataFrame(examples)
score_dataframe(sample_df)